In [ ]:
import os
import random
import pickle
import numpy as np
import tensorflow as tf
from keras import backend as K
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, BatchNormalization
from keras.layers import Activation, Flatten, Dense, Dropout
from keras.layers import ELU
from keras.initializers import glorot_uniform
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
import datetime
import matplotlib.pyplot as plt
import cupy as cp

# Set random seeds for reproducibility
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

print("✓ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
import os
import pickle
import numpy as np
from natsort import natsorted
import glob
from tqdm import tqdm
import zipfile
import shutil
from sklearn.preprocessing import LabelEncoder

# Configuration
max_matrix_len = 2000
max_time = 60

def check_gpu():
    """Check if GPU is available"""
    try:
        print("GPU Information:")
        print(f"GPU Available: {cp.cuda.is_available()}")
        if cp.cuda.is_available():
            print(f"GPU Device: {cp.cuda.Device()}")
            print(f"GPU Memory: {cp.cuda.Device().mem_info[1] / 1e9:.2f} GB")
        print()
    except Exception as e:
        print(f"GPU check failed: {e}")
        print("Falling back to CPU mode\n")
        
def setup_folders():
    """Create necessary folders for processing"""
    folders = ['uploaded_data', 'output']
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
    return folders

def extract_zip_files(input_path='/kaggle/input/datasets/hphglinh/mac-new-ccs/ccs_50labels500_mac_new'):
    """Extract txt files from zip in Kaggle input directory"""
    print("Looking for ZIP files in Kaggle input directory...")
    
    # Find all zip files in input directory
    zip_files = glob.glob(f'{input_path}/**/*.zip', recursive=True)
    
    if len(zip_files) == 0:
        print(f"No ZIP files found in {input_path}")
        # Try to find txt files directly
        txt_files = glob.glob(f'{input_path}/**/*.txt', recursive=True)
        if len(txt_files) > 0:
            print(f"Found {len(txt_files)} .txt files directly in input")
            return txt_files
        return []
    
    # Extract all zip files
    for zip_file in zip_files:
        print(f"Extracting {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('uploaded_data')
        print("Extraction complete!")
    
    # Find all txt files (including in subdirectories)
    txt_files = glob.glob('uploaded_data/**/*.txt', recursive=True)
    print(f"Found {len(txt_files)} .txt files")
    return txt_files

def winlap(instance, max_matrix_len, max_time=60):
    """Extract overlapping time-amplitude matrix features"""
    times = []
    sizes = []

    for line in instance:
        try:
            time, size = line.split()
            times.append(float(time))
            size = size.split('\n')[0]
            sizes.append(int(float(size)))
        except:
            continue

    window_size = max_time / max_matrix_len
    sliding_interval = window_size / 2
    max_matrix_len = max_matrix_len * 2
    feature = [0] * max_matrix_len * 2

    for i in range(len(sizes)):
        idx = int(times[i] / max_time * max_matrix_len)
        if idx >= max_matrix_len:
            idx = max_matrix_len - 1

        # if sizes[i] < 0:  # incoming
        feature[idx] += 1
        feature[idx + max_matrix_len] += -sizes[i]
        pre_idx = max(idx-1, 0)
        feature[pre_idx] += 1
        feature[pre_idx + max_matrix_len] += -sizes[i]

    return feature

def process_file(file):
    """Process a single file and return feature vector with label"""
    with open(file, 'r') as f:
        instance = f.readlines()

    # Extract label from filename (format: label_index.txt -> extract "label")
    filename = os.path.basename(file)
    try:
        # Split by underscore and take all parts except the last one (which is index.txt)
        parts = filename.rsplit('_', 1)  # Split from right, only once
        label = parts[0]  # Everything before the last underscore is the label
    except:
        label = "unknown"

    feature = winlap(instance, max_matrix_len, max_time)
    
    return feature, label

def extract_features(files):
    """Extract features from all files"""
    print(f"Processing {len(files)} files...")
    X = []
    y = []

    for file in tqdm(files, desc="Extracting features"):
        try:
            feature_vector, label = process_file(file)
            X.append(feature_vector)
            y.append(label)
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
            continue

    print("Converting to numpy arrays...")
    X = np.array(X)
    
    # Encode string labels to integers (0, 1, 2, ...)
    print("\nEncoding labels to integers...")
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    y = y_encoded.reshape(-1, 1)
    
    # Print label mapping
    print(f"\nFound {len(label_encoder.classes_)} unique labels:")
    print("\nLabel mapping:")
    for i, label in enumerate(label_encoder.classes_):
        count = np.sum(y == i)
        print(f"  {i}: {label} ({count} samples)")

    return X, y, label_encoder

def save_to_pickle(data, filename):
    """Save data to pickle file"""
    print(f"Saving to {filename}...")
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved successfully!")

# Main execution
if __name__ == "__main__":
    print("=== Feature Extraction for Kaggle ===\n")
    
    # Check GPU availability
    use_gpu = False
    try:
        check_gpu()
        use_gpu = cp.cuda.is_available()
    except:
        print("CuPy not available, using CPU mode\n")
        use_gpu = False
        

    # Clean old data
    if os.path.exists('uploaded_data'):
        shutil.rmtree('uploaded_data')
    if os.path.exists('output'):
        shutil.rmtree('output')

    # Setup
    setup_folders()

    # Extract files from Kaggle input
    txt_files = extract_zip_files()

    if len(txt_files) == 0:
        print("No .txt files found! Please check your input dataset.")
    else:
        # Sort files naturally
        txt_files = natsorted(txt_files)

        # Extract features and encode labels
        X, y, label_encoder = extract_features(txt_files)

        print(f"\nFeature matrix shape: {X.shape}")
        print(f"Label matrix shape: {y.shape}")
        print(f"Label range: {y.min()} to {y.max()}")

        # Save to pickle in /kaggle/working (Kaggle's output directory)
        X_path = '/kaggle/working/x_train.pkl'
        y_path = '/kaggle/working/y_train.pkl'
        encoder_path = '/kaggle/working/label_encoder.pkl'

        save_to_pickle(X, X_path)
        save_to_pickle(y, y_path)
        save_to_pickle(label_encoder, encoder_path)

        print("\n=== Processing Complete! ===")
        print(f"Files saved to /kaggle/working/")
        print(f"- {X_path}")
        print(f"- {y_path}")
        print(f"- {encoder_path}")
        print("\nThese files are now available as output in your Kaggle notebook.")
        print("\nTo decode labels back to names:")
        print("  label_encoder.inverse_transform([0, 1, 2, ...])")

SET DATAPATH


In [ ]:
# Example paths (CHANGE THESE to match your dataset):
DATA_PATH = "/kaggle/working"  # Change this!
X_PATH = f"{DATA_PATH}/x_train.pkl"
Y_PATH = f"{DATA_PATH}/y_train.pkl"

# Verify files exist
import os
if os.path.exists(X_PATH):
    print(f"✓ Found X file: {X_PATH}")
else:
    print(f"❌ X file not found: {X_PATH}")
    print(f"Available files in {DATA_PATH}:")
    print(os.listdir(DATA_PATH))

if os.path.exists(Y_PATH):
    print(f"✓ Found y file: {Y_PATH}")
else:
    print(f"❌ y file not found: {Y_PATH}")

DEFINE DKF MODEL

In [ ]:

class DKFNet:
    @staticmethod
    def build(input_shape, classes):
        model = Sequential()
        #Block1
        filter_num = ['None',32,64,128,256, 512]
        kernel_size = ['None',8,8,8,8,8]
        conv_stride_size = ['None',1,1,1,1,1]
        pool_stride_size = ['None',4,4,4,4,4]
        pool_size = ['None',8,8,8,8,8]

        # ======= First Block =======
        model.add(Conv1D(filters=filter_num[1], kernel_size=kernel_size[1], input_shape=(8000,1),
                         strides=conv_stride_size[1], padding='same',
                         name='block1_conv1'))
        model.add(BatchNormalization(axis=-1))
        model.add(ELU(alpha=1.0, name='block1_adv_act1'))
        model.add(Conv1D(filters=filter_num[1], kernel_size=kernel_size[1],
                         strides=conv_stride_size[1], padding='same',
                         name='block1_conv2'))
        model.add(BatchNormalization(axis=-1))
        model.add(ELU(alpha=1.0, name='block1_adv_act2'))
        model.add(MaxPooling1D(pool_size=pool_size[1], strides=pool_stride_size[1],
                               padding='same', name='block1_pool'))
        model.add(Dropout(0.2, name='block1_dropout'))

        # ======= Second Block =======
        model.add(Conv1D(filters=filter_num[2], kernel_size=kernel_size[2],
                         strides=conv_stride_size[2], padding='same',
                         name='block2_conv1'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block2_act1'))
        model.add(Conv1D(filters=filter_num[2], kernel_size=kernel_size[2],
                         strides=conv_stride_size[2], padding='same',
                         name='block2_conv2'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block2_act2'))
        model.add(MaxPooling1D(pool_size=pool_size[2], strides=pool_stride_size[3],
                               padding='same', name='block2_pool'))
        model.add(Dropout(0.2, name='block2_dropout'))

        # ======= Third Block =======
        model.add(Conv1D(filters=filter_num[3], kernel_size=kernel_size[3],
                         strides=conv_stride_size[3], padding='same',
                         name='block3_conv1'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block3_act1'))
        model.add(Conv1D(filters=filter_num[3], kernel_size=kernel_size[3],
                         strides=conv_stride_size[3], padding='same',
                         name='block3_conv2'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block3_act2'))
        model.add(MaxPooling1D(pool_size=pool_size[3], strides=pool_stride_size[3],
                               padding='same', name='block3_pool'))
        model.add(Dropout(0.2, name='block3_dropout'))

        # ======= Fourth Block =======
        model.add(Conv1D(filters=filter_num[4], kernel_size=kernel_size[4],
                         strides=conv_stride_size[4], padding='same',
                         name='block4_conv1'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block4_act1'))
        model.add(Conv1D(filters=filter_num[4], kernel_size=kernel_size[4],
                         strides=conv_stride_size[4], padding='same',
                         name='block4_conv2'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block4_act2'))
        model.add(MaxPooling1D(pool_size=pool_size[4], strides=pool_stride_size[4],
                               padding='same', name='block4_pool'))
        model.add(Dropout(0.2, name='block4_dropout'))

        # ======= Fifth Block =======
        model.add(Conv1D(filters=filter_num[5], kernel_size=kernel_size[5],
                         strides=conv_stride_size[5], padding='same',
                         name='block5_conv1'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block5_act1'))
        model.add(Conv1D(filters=filter_num[5], kernel_size=kernel_size[5],
                         strides=conv_stride_size[5], padding='same',
                         name='block5_conv'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='block5_act2'))
        model.add(MaxPooling1D(pool_size=pool_size[5], strides=pool_stride_size[5],
                               padding='same', name='block5_pool'))
        model.add(Dropout(0.2, name='block5_dropout'))

        # ======= FC Block =======
        model.add(Flatten(name='flatten'))
        model.add(Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc1'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='fc1_act'))
        model.add(Dropout(0.7, name='fc1_dropout'))

        model.add(Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc2'))
        model.add(BatchNormalization())
        model.add(Activation('relu', name='fc2_act'))
        model.add(Dropout(0.5, name='fc2_dropout'))

        model.add(Dense(classes, kernel_initializer=glorot_uniform(seed=0), name='fc3'))
        model.add(Activation('softmax', name="softmax"))
        return model

DEFINE CUSTOM CALLBACK

In [ ]:
class EpochLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"\n📊 Epoch {epoch + 1}: "
              f"loss={logs['loss']:.4f}, accuracy={logs['accuracy']:.4f}, "
              f"val_loss={logs['val_loss']:.4f}, val_accuracy={logs['val_accuracy']:.4f}")

print("✓ Custom callback defined!")

CONFIGURATION

In [ ]:
# CHANGE THESE SETTINGS AS NEEDED
MODEL_NAME = 'dkf'  # Options: 'dkf' or 'tiktok' (for DF model)
NB_EPOCH = 10
BATCH_SIZE = 32
VERBOSE = 2
NB_CLASSES = 50

# Set sequence length based on model
if MODEL_NAME == 'dkf':
    LENGTH = 5000
elif MODEL_NAME == 'tiktok':
    LENGTH = 5000
else:
    LENGTH = 8000  # default

INPUT_SHAPE = (LENGTH, 1)
OPTIMIZER = Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999, epsilon=1e-08)

print("=" * 50)
print("⚙️  CONFIGURATION")
print("=" * 50)
print(f"Model Type: {MODEL_NAME.upper()}")
print(f"Sequence Length: {LENGTH}")
print(f"Number of Classes: {NB_CLASSES}")
print(f"Epochs: {NB_EPOCH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Optimizer: Adamax (lr=0.002)")
print("=" * 50)

LOAD AND PREPARE DATA

In [ ]:
import pickle
import numpy as np
from sklearn.model_selection import train_test_split

def LoadData(X_path, y_path):
    """Load pickle files and split into train/valid/test sets with stratification"""
    print("\n📂 Loading data from pickle files...")
    
    with open(X_path, 'rb') as f:
        X = pickle.load(f)
    with open(y_path, 'rb') as f:
        y = pickle.load(f)
    
    print(f"✓ Loaded X: {X.shape}, y: {y.shape}")
    
    # Flatten y for stratification
    y_flat = y.ravel()
    
    # Check class distribution
    unique_classes, counts = np.unique(y_flat, return_counts=True)
    print(f"\n📊 Total unique classes: {len(unique_classes)}")
    print(f"   Class range: {unique_classes.min()} to {unique_classes.max()}")
    
    # Find classes with very few samples
    min_samples = counts.min()
    print(f"   Minimum samples per class: {min_samples}")
    
    if min_samples < 5:
        print(f"\n⚠️ Warning: Some classes have very few samples (<5)!")
        print("   Classes with few samples:")
        for cls, count in zip(unique_classes, counts):
            if count < 5:
                print(f"      Class {cls}: {count} samples")
    
    # Split data: 60% train, 20% validation, 20% test
    # Using stratify to maintain class distribution
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, 
        test_size=0.3, 
        random_state=42,
        stratify=y_flat  # Ensure proportional class distribution
    )
    
    # Split temp into validation and test
    y_temp_flat = y_temp.ravel()
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, 
        test_size=0.5, 
        random_state=42,
        stratify=y_temp_flat  # Ensure proportional class distribution
    )
    
    print("\n📊 Data Split:")
    print(f"  Train: X={X_train.shape}, y={y_train.shape}")
    print(f"  Valid: X={X_valid.shape}, y={y_valid.shape}")
    print(f"  Test:  X={X_test.shape}, y={y_test.shape}")
    
    # Verify class distribution in each split
    print("\n📊 Class distribution verification:")
    train_classes = len(np.unique(y_train))
    valid_classes = len(np.unique(y_valid))
    test_classes = len(np.unique(y_test))
    
    print(f"  Train: {train_classes} unique classes")
    print(f"  Valid: {valid_classes} unique classes")
    print(f"  Test:  {test_classes} unique classes")
    
    if train_classes != valid_classes or train_classes != test_classes:
        print("\n⚠️ Warning: Class distribution is not equal across splits!")
        print("   This may happen if some classes have very few samples.")
    else:
        print("\n✓ All splits have the same number of classes!")
    
    return X_train, y_train, X_valid, y_valid, X_test, y_test

# Load the data using the paths defined earlier
X_train, y_train, X_valid, y_valid, X_test, y_test = LoadData(X_PATH, Y_PATH)

DATA PREPROCESSING

In [ ]:

print("\n🔄 Preprocessing data...")

# Convert to float32
X_train = X_train.astype('float32')
X_valid = X_valid.astype('float32')
X_test = X_test.astype('float32')
y_train = y_train.astype('float32')
y_valid = y_valid.astype('float32')
y_test = y_test.astype('float32')

# Reshape to add channel dimension [samples, length, channels]
X_train = X_train[:, :, np.newaxis]
X_valid = X_valid[:, :, np.newaxis]
X_test = X_test[:, :, np.newaxis]

print("After reshaping:")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Valid: X={X_valid.shape}, y={y_valid.shape}")
print(f"  Test:  X={X_test.shape}, y={y_test.shape}")

# Convert labels to one-hot encoding
y_train = to_categorical(y_train.ravel(), NB_CLASSES)
y_valid = to_categorical(y_valid.ravel(), NB_CLASSES)
y_test = to_categorical(y_test.ravel(), NB_CLASSES)

print("\nAfter one-hot encoding:")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Valid: X={X_valid.shape}, y={y_valid.shape}")
print(f"  Test:  X={X_test.shape}, y={y_test.shape}")
print("✓ Data preprocessing complete!")

BUILD MODEL

In [ ]:
print("\n🏗️  Building model...")

if MODEL_NAME == 'dkf':
    print("Building DKF model...")
    model = DKFNet.build(input_shape=INPUT_SHAPE, classes=NB_CLASSES)
elif MODEL_NAME == 'tiktok':
    print("Building DF model...")
    model = DKFNet.build(input_shape=INPUT_SHAPE, classes=NB_CLASSES)
else:
    raise ValueError(f"Unknown model name: {MODEL_NAME}")

model.compile(
    loss="categorical_crossentropy",
    optimizer=OPTIMIZER,
    metrics=["accuracy"]
)

print("✓ Model built and compiled!")
print("\n📋 Model Summary:")
model.summary()

TRAIN MODEL

In [ ]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

epoch_logger = EpochLogger()

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=30,
    verbose=VERBOSE,
    validation_data=(X_valid, y_valid),
    callbacks=[epoch_logger]
)

print("\n✓ Training complete!")

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
import pickle

# -------------------------------
# Model evaluation and reporting
# -------------------------------
print("\n" + "="*50)
print("📊 MODEL EVALUATION")
print("="*50)

VERBOSE = 1  # or 0 to disable logging

# 1️⃣ Evaluate on training set
print("\nEvaluating on training set...")
score_train = model.evaluate(X_train, y_train, verbose=VERBOSE)

# 2️⃣ Evaluate on test set
print("\nEvaluating on test set...")
score_test = model.evaluate(X_test, y_test, verbose=VERBOSE)

# 3️⃣ Convert predictions to class labels
if y_test.ndim > 1 and y_test.shape[1] > 1:
    # One-hot encoding -> take argmax
    y_pred_classes = np.argmax(model.predict(X_test), axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
else:
    # Binary or single label
    y_pred_classes = (model.predict(X_test) > 0.5).astype(int).flatten()
    y_true_classes = y_test.flatten()

# 4️⃣ Load label encoder to get class names
try:
    with open('/kaggle/working/label_encoder.pkl', 'rb') as f:
        label_encoder = pickle.load(f)
    
    # Use all possible classes from label encoder
    all_classes = list(range(len(label_encoder.classes_)))
    target_names = [f"{i}: {label_encoder.classes_[i]}" for i in all_classes]
    
    print("\n📄 Classification Report on Test Set:")
    print(classification_report(
        y_true_classes, 
        y_pred_classes, 
        labels=all_classes,  # Specify all possible labels
        target_names=target_names,
        zero_division=0  # Avoid warnings for classes with no predictions
    ))
    
except FileNotFoundError:
    # Fallback if label encoder not found
    print("\n⚠️ Label encoder not found. Using numeric labels only.")
    all_classes = list(range(NB_CLASSES))
    target_names = [str(cls) for cls in all_classes]
    
    print("\n📄 Classification Report on Test Set:")
    print(classification_report(
        y_true_classes, 
        y_pred_classes,
        labels=all_classes,
        target_names=target_names,
        zero_division=0
    ))

# 5️⃣ Accuracy per class (only for classes that appear in test set)
print("\n📊 Accuracy per class (classes present in test set):")
classes_in_test = np.unique(y_true_classes)
for cls in sorted(classes_in_test):
    idx = np.where(y_true_classes == cls)
    cls_acc = np.mean(y_true_classes[idx] == y_pred_classes[idx])
    
    # Get class name if label encoder is available
    try:
        class_name = label_encoder.classes_[cls]
        print(f" - Class {cls} ({class_name}): {cls_acc*100:.2f}%")
    except:
        print(f" - Class {cls}: {cls_acc*100:.2f}%")

# 6️⃣ Show which classes are missing from test set
all_possible_classes = set(range(NB_CLASSES))
classes_in_test_set = set(classes_in_test)
missing_classes = all_possible_classes - classes_in_test_set

if missing_classes:
    print(f"\n⚠️ Warning: {len(missing_classes)} classes not present in test set:")
    for cls in sorted(missing_classes):
        try:
            class_name = label_encoder.classes_[cls]
            print(f"   - Class {cls} ({class_name})")
        except:
            print(f"   - Class {cls}")

# -------------------------------
# Final results summary
# -------------------------------
print("\n" + "="*50)
print("📈 FINAL RESULTS")
print("="*50)
print(f"Epochs: {NB_EPOCH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Model: {MODEL_NAME.upper()}")
print(f"Total Classes: {NB_CLASSES}")
print(f"Classes in Test Set: {len(classes_in_test)}")

print(f"\nMetrics: {model.metrics_names}")
print(f"\n🎯 Training Results:")
print(f"   Loss: {score_train[0]:.4f}")
print(f"   Accuracy: {score_train[1]:.4f} ({score_train[1]*100:.2f}%)")

print(f"\n🎯 Test Results:")
print(f"   Loss: {score_test[0]:.4f}")
print(f"   Accuracy: {score_test[1]:.4f} ({score_test[1]*100:.2f}%)")

print("="*50)